# BỘ PHÂN XỬ CẶP — huấn luyện + đánh giá

Bộ chấm hiện tại chấm **từng cặp (câu hỏi, văn bản) độc lập** — chưa bao giờ nhìn hai ứng viên
cạnh nhau. Bộ phân xử nhìn cả hai rồi trả lời **"cái nào đúng hơn"**.

```
input : [CLS] cau hoi [SEP] [A] van ban A [B] van ban B [SEP]
nhan  : 1 neu A la gold · mat mat BCEWithLogits
```

## NGƯỠNG ĐÃ KHOÁ — đọc TRƯỚC khi xem kết quả

Trên dải **margin < 0,005** của dev300: **65 cặp · 46 đang đúng · 19 cứu được · hoà vốn 70,8%**.

n=65 nên sai số chuẩn ±5,7 điểm — đúng 71% không phân biệt được với hoà vốn. Để vượt **có ý
nghĩa** (1 phía, p<0,05):

> **NHẬN nếu đúng ≥ 53/65 = 81,5%.**
> Dưới ⇒ **ĐÓNG, không chạy đề thi, không nộp. Không được đổi ngưỡng sau khi thấy số.**

Dư địa: phân xử hoàn hảo = +19 câu (+6,3 điểm dev300) · đạt đúng ngưỡng = +7 câu (+2,3 điểm).

## Hai chi tiết bắt buộc, bỏ là hỏng âm thầm
1. **Thứ tự đã đảo ngẫu nhiên** trong tập train (tỉ lệ nhãn 0,498). Không đảo → model học "luôn chọn A".
2. **Suy luận phải chấm CẢ HAI CHIỀU** (A,B) và (B,A) rồi lấy trung bình. Ô 3 đã làm.

Bỏ ngay nếu val acc không vượt 0,50 (ngẫu nhiên) ở mốc 500 bước.

---

## SỬA 11/09 — lượt chạy đầu KHÔNG HỢP LỆ, đã vá 2 lỗi

**Lỗi 1 — dùng lại đầu phân loại cũ.** `Vietnamese_Reranker` đã có sẵn đầu `num_labels=1`,
nhưng đó là đầu chấm *"đoạn này liên quan tới câu hỏi không"* với logit biên độ ±10. Nhiệm vụ
mới là *"A hay B đúng hơn"* — không gian ngữ nghĩa khác. Giữ đầu cũ ⇒ BCE khởi đầu ở loss
**~2,5** thay vì `ln2 = 0,693`, và model phải **bỏ học** thang đo cũ trước. Nay reset đầu
phân loại + assert logit khởi đầu gần 0.

**Lỗi 2 — gate đặt trong warmup (lần thứ HAI).** Warmup = **2.112 bước batch**, mà gate đặt ở
bước **500 = 24% của warmup** ⇒ cắt một model gần như chưa học gì. Nay moc gate tính từ
`total_steps`: **1,5× số bước warmup**, không bao giờ cố định.

Số của lượt đầu (60,0% trên ô quyết định) **không phải bằng chứng về hướng này** — nó là bằng
chứng lượt chạy bị hỏng.


In [ ]:
import os, sys, json, time, random, math, hashlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"   # PHAI dat TRUOC import torch
import numpy as np, torch, torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

assert torch.cuda.is_available(), "CHUA BAT GPU -> Settings > Accelerator > GPU T4"
print(f"  GPU: {torch.cuda.get_device_name(0)}")

ROOT="/kaggle/input"
def find(name):
    for r,_,fs in os.walk(ROOT):
        if name in fs: return os.path.join(r,name)
    raise FileNotFoundError(f"KHONG THAY {name} duoi {ROOT}")

TRAIN = find("pairset_hard.jsonl")
EVALF = find("pairset_dev300.jsonl")

# quy tac 8: nhan dien bang RUOT, khong bang TEN
SIG = {"pairset_hard.jsonl":   ("f49bc2080ea63bcbe898155882b4e3c8", 22028),
       "pairset_dev300.jsonl": ("54373b275fb0aee28a53e8db0d022d80",   254)}
for p in (TRAIN, EVALF):
    h = hashlib.md5(open(p,"rb").read()).hexdigest()
    n = sum(1 for _ in open(p, encoding="utf-8"))
    want_h, want_n = SIG[os.path.basename(p)]
    print(f"  {os.path.basename(p):<22} md5={h[:12]}  n={n:,}")
    assert h==want_h and n==want_n, f"SAI FILE {p}\n  co {h} n={n}\n  can {want_h} n={want_n}"

# seed day du — lo tai lap phat hien 10/09
torch.manual_seed(0); torch.cuda.manual_seed_all(0); random.seed(0); np.random.seed(0)
torch.use_deterministic_algorithms(True, warn_only=True)

MODEL   = "AITeamVN/Vietnamese_Reranker"
MAXLEN  = 1536      # input dai toi da 3.817 ky tu -> ~1.272 token. 1024 la KHONG DU.
BATCH   = 1         # cap dai gap doi listwise -> giu 1
ACC     = 16
LR      = 1e-5
EPOCHS  = 1
HOLDOUT = 800       # 800 cap kiem tra (=200 nhom), phan con lai huan luyen

rows=[json.loads(l) for l in open(TRAIN,encoding="utf-8")]
random.Random(0).shuffle(rows)
val, tr = rows[:HOLDOUT], rows[HOLDOUT:]
print(f"\n  {len(tr):,} cap huan luyen · {len(val)} cap kiem tra")
print(f"  ti le nhan=1 tren tap train: {np.mean([r['label'] for r in tr]):.3f}  (phai ~0.5)")

tok = AutoTokenizer.from_pretrained(MODEL)
def enc(batch):
    """cau hoi la segment 1; hai van ban ghep vao segment 2 voi moc [A]/[B]."""
    qs = [b["question"] for b in batch]
    ts = [f"[A] {b['A']} [B] {b['B']}" for b in batch]
    e = tok(qs, ts, truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
    y = torch.tensor([float(b["label"]) for b in batch])
    return e, y

class DS(Dataset):
    def __init__(s, r): s.r=r
    def __len__(s): return len(s.r)
    def __getitem__(s, i): return s.r[i]


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).cuda()

# ===== SUA 11/09 — LOI 1: phai KHOI TAO LAI dau phan loai =====
# `AITeamVN/Vietnamese_Reranker` DA CO san dau num_labels=1, nhung do la dau cham
# "doan nay lien quan toi cau hoi khong" — no xuat logit bien do lon (+-10).
# Nhiem vu moi la "A hay B dung hon", mot khong gian ngu nghia KHAC.
# Giu dau cu => BCE bat dau o loss ~2.5 thay vi ln2=0.693, va model phai
# BO HOC thang do cu truoc khi hoc duoc viec moi. Lan chay dau bi dinh dung loi nay.
_head = model.classifier
for _m in _head.modules():
    if isinstance(_m, torch.nn.Linear):
        _m.weight.data.normal_(mean=0.0, std=0.02)
        if _m.bias is not None: _m.bias.data.zero_()
with torch.no_grad():
    _e,_y = enc(val[:4])
    _l = model(**{k:v.cuda() for k,v in _e.items()}).logits.view(-1)
print(f"  dau phan loai da reset · logit mau: {[round(x,3) for x in _l.float().cpu().tolist()]}")
print(f"  (phai gan 0. Neu con bien do lon (+-5) thi reset KHONG an -> dung lai)")
assert _l.abs().max().item() < 2.0, "reset dau phan loai KHONG an -> dung, dung chay tiep"

model.gradient_checkpointing_enable()          # KHONG CO -> OOM
print(f"tham so: {sum(p.numel() for p in model.parameters())/1e9:.3f}B · grad ckpt: BAT")
print(f"VRAM sau khi nap: {torch.cuda.memory_allocated()/2**30:.2f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.2f} GB")

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
dl  = DataLoader(DS(tr), batch_size=BATCH, shuffle=True, collate_fn=enc, drop_last=True)
total = len(dl)*EPOCHS
WARM_B = int((total//ACC+1) * 0.1) * ACC          # so buoc BATCH cua giai doan warmup
GATE   = min(int(WARM_B*1.5), total//3)           # gate SAU warmup, khong phai o buoc 500
print(f"warmup = {WARM_B:,} buoc batch · dat gate o buoc {GATE:,} ({GATE/WARM_B:.1f}x warmup)")
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=total//ACC+1, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")

def val_acc(n=None):
    model.eval(); ok=0; V = val[:n] if n else val
    with torch.no_grad():
        for i in range(0, len(V), 8):
            e,y = enc(V[i:i+8])
            with torch.autocast("cuda", dtype=torch.float16):
                s = model(**{k:v.cuda() for k,v in e.items()}).logits.view(-1)
            ok += ((s.float().cpu() > 0).int() == y.int()).sum().item()
    model.train(); return ok/len(V)

v0 = val_acc(400)
print(f"\ndo chinh xac kiem tra TRUOC huan luyen: {v0:.4f}  (ngau nhien = 0.50)")

model.train(); t0=time.time(); losses=[]
for step, (e, y) in enumerate(dl, 1):
    with torch.autocast("cuda", dtype=torch.float16):
        logit = model(**{k:v.cuda() for k,v in e.items()}).logits.view(-1)
        loss  = Fn.binary_cross_entropy_with_logits(logit, y.cuda()) / ACC
    scaler.scale(loss).backward()
    losses.append(loss.item()*ACC)
    if step % ACC == 0:
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
    if step == 1:
        pk = torch.cuda.max_memory_allocated()/2**30
        tot= torch.cuda.get_device_properties(0).total_memory/2**30
        print(f"  buoc 1 OK · dinh VRAM {pk:.2f}/{tot:.2f} GB · du dia {tot-pk:.2f} GB", flush=True)
        if tot-pk < 1.0: print("  !!! DU DIA <1 GB -> ha MAXLEN xuong 1280")
    if step % 200 == 0:
        el=(time.time()-t0)/60
        print(f"  buoc {step}/{total} · loss {np.mean(losses[-200:]):.4f} · {el:.1f} phut "
              f"· con ~{el/step*(total-step):.0f} phut", flush=True)
    if step == GATE:
        v5 = val_acc(400)
        print(f"  >>> CHOT buoc {GATE} (= {GATE/WARM_B:.1f}x warmup): kiem tra {v5:.4f} (truoc {v0:.4f})")
        # SUA 11/09 — LOI 2: gate phai dat SAU warmup.
        #   Lan chay dau gate o buoc 500 = 24% cua warmup (warmup = 2.112 buoc batch),
        #   tuc model gan nhu CHUA HOC GI. Day la lan thu HAI gate dat sai cho:
        #   09/09 gate bang loss trong warmup, 11/09 gate bang val acc trong warmup.
        #   Tu nay: moc gate = 1.5x so buoc warmup, tinh tu total_steps, khong bao gio co dinh.
        if v5 <= 0.52:
            print(f"  !!! KHONG HON NGAU NHIEN sau {GATE} buoc -> DUNG."); break

vf = val_acc()
print(f"\nxong · kiem tra SAU huan luyen ({len(val)} cap): {vf:.4f}")
model.save_pretrained("/kaggle/working/judge_pair"); tok.save_pretrained("/kaggle/working/judge_pair")
print("da luu /kaggle/working/judge_pair — TAI VE TRUOC KHI DONG PHIEN")


In [ ]:
# ===== DANH GIA tren dev300 — cham CA HAI CHIEU roi lay trung binh =====
import json
E = [json.loads(l) for l in open(EVALF, encoding="utf-8")]
print(f"{len(E)} cap danh gia\n")

model.eval(); pred = {}
with torch.no_grad():
    for i in range(0, len(E), 8):
        b = E[i:i+8]
        # chieu thuan (A,B) va chieu nghich (B,A) — triet tieu thien lech vi tri
        e1,_ = enc([{**r} for r in b])
        e2,_ = enc([{**r, "A": r["B"], "B": r["A"]} for r in b])
        with torch.autocast("cuda", dtype=torch.float16):
            s1 = model(**{k:v.cuda() for k,v in e1.items()}).logits.view(-1).float().cpu()
            s2 = model(**{k:v.cuda() for k,v in e2.items()}).logits.view(-1).float().cpu()
        for r, a, b_ in zip(b, s1.tolist(), s2.tolist()):
            pred[r["qid"]] = (a - b_) / 2      # >0 => chon A (hang 1)

def bao_cao(t, ten):
    sub = [r for r in E if abs(r["margin"]) < t]
    if not sub: return
    n = len(sub)
    hoa_von = sum(r["label"] for r in sub) / n
    dung = sum(int((pred[r["qid"]] > 0) == bool(r["label"])) for r in sub)
    cuu  = sum(1 for r in sub if r["label"]==0 and pred[r["qid"]] <= 0)
    hong = sum(1 for r in sub if r["label"]==1 and pred[r["qid"]] <= 0)
    print(f"{ten}  n={n}")
    print(f"   bo phan xu dung : {dung}/{n} = {dung/n:.1%}")
    print(f"   hoa von         : {hoa_von:.1%}")
    print(f"   cuu {cuu} · hong {hong} · NET {cuu-hong:+d} cau = {(cuu-hong)/300*100:+.2f} diem dev300\n")

print("="*66)
bao_cao(0.005, "[O QUYET DINH] dai margin < 0.005")
bao_cao(0.02,  "[tham khao]    dai margin < 0.02 ")
bao_cao(9e9,   "[tham khao]    toan bo top-2     ")
print("="*66)

sub = [r for r in E if abs(r["margin"]) < 0.005]
dung = sum(int((pred[r["qid"]] > 0) == bool(r["label"])) for r in sub)
print(f"NGUONG KHOA: can >= 53/65 = 81.5%   ·   dat duoc: {dung}/{len(sub)} = {dung/len(sub):.1%}")
print(">>> " + ("DAT -> chay de thi bang bo phan xu, roi nop."
                if dung >= 53 else
                "KHONG DAT -> DONG. Giu bai 0,711. Khong chay de thi, khong nop."))
json.dump({q: float(v) for q, v in pred.items()},
          open("/kaggle/working/judge_pred_dev300.json", "w"))
print("\nda luu /kaggle/working/judge_pred_dev300.json")
